In [3]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, train_test_split


# Load data
data = pd.read_csv(
    r"C:\Users\User\Documents\GitHub\spam-detector\data\processed\spam_unified1.csv"
)


# Features
X = data[
    [
        "text",
        "url_count",
        "exclamation_count",
        "capital_ratio",
        "digit_ratio",
        "message_length"
    ]
]

# Target
y = data["label"]


# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [4]:

# Apply TF-IDF only to text
preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(),
            "text"
        )
    ],
    remainder="passthrough"
)


# Full pipeline
pipeline = Pipeline([
    ("features", preprocessor),
    ("model", MultinomialNB())
])


# Hyperparameter grid
param_grid = {
    "features__tfidf__ngram_range": [(1, 1), (1, 2)],
    "features__tfidf__min_df": [1, 2, 5],
    "features__tfidf__max_features": [10000, 15000, 20000],
    "features__tfidf__stop_words": [None, "english"],
    "model__alpha": [0.1, 0.5, 1.0, 2.0]
}



In [5]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)


In [6]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...inomialNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'features__tfidf__max_features': [10000, 15000, ...], 'features__tfidf__min_df': [1, 2, ...], 'features__tfidf__ngram_range': [(1, ...), (1, ...)], 'features__tfidf__stop_words': [None, 'english'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` param

In [7]:
print("Best parameters:")
print(grid_search.best_params_)

print("Best CV F1:")
print(grid_search.best_score_)

Best parameters:
{'features__tfidf__max_features': 20000, 'features__tfidf__min_df': 5, 'features__tfidf__ngram_range': (1, 2), 'features__tfidf__stop_words': 'english', 'model__alpha': 0.1}
Best CV F1:
0.924485280405935


In [8]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9220786548552729
Precision: 0.9085385700336335
Recall: 0.9396319569120287
F1: 0.9238237078713664

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92      8811
           1       0.91      0.94      0.92      8912

    accuracy                           0.92     17723
   macro avg       0.92      0.92      0.92     17723
weighted avg       0.92      0.92      0.92     17723


Confusion Matrix:
[[7968  843]
 [ 538 8374]]


In [11]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)

best_model = grid_search.best_estimator_

joblib.dump(best_model, "models/spam_clf.joblib")

print("Model saved successfully!")

Model saved successfully!


In [12]:
messages = [
    "Congratulations! You have won $1000. Click here now!",
    "Hey, are you coming to class tomorrow?",
    "URGENT! Claim your FREE prize now!!!",
]

predictions = best_model.predict(
    pd.DataFrame({
        "text": messages,
        "url_count": [0, 0, 0],
        "exclamation_count": [3, 0, 3],
        "capital_ratio": [0.10, 0.05, 0.30],
        "digit_ratio": [0.05, 0.00, 0.05],
        "message_length": [50, 30, 35]
    })
)

print(predictions)

[1 0 1]
